# Think like a hacker: Python security gotchas. 
# Code defensively

## 1. malware hidden in supply chain - import an untrusted module

In [ ]:
# ordinal_v1.py
# Utility to return an English string ordinal for any number
# eg: 1st, 2nd, 3rd, 4th, etc.

def ordinal(n: int) -> str:
    """Return an integer as its ordinal string (1st, 2nd, 3rd, etc.)."""
    if not isinstance(n, int):
        raise TypeError("n must be an integer")

    abs_n = abs(n)
    last_two = abs_n % 100

    if 11 <= last_two <= 13:
        suffix = "th"
    else:
        suffix = {1: "st", 2: "nd", 3: "rd"}.get(abs_n % 10, "th")

    return f"{n}{suffix}"

if __name__ == '__main__':
    print(1,ordinal(1))    # "1st"
    print(2,ordinal(2))    # "2nd"
    print(3,ordinal(3))    # "3rd"
    print(4,ordinal(4))    # "4th"
    print(11,ordinal(11))   # "11th"
    print(22,ordinal(22))   # "22nd"
    print(113,ordinal(113))  # "113th"

In [1]:
!python3 ordinal_v1.py


1 1st
2 2nd
3 3rd
4 4th
11 11th
22 22nd
113 113th


In [3]:
from ordinal_v1 import ordinal

my_position=1
your_position=2
print( f"In this race, I am the {ordinal(my_position)} and you are the {ordinal(your_position)} at this time")

In this race, I am the 1st and you are the 2nd at this time


### Now let's poison `ordinal` with malware

In [ ]:
# ordinal_v2.py
# Utility to return an English string ordinal for any number
# eg: 1st, 2nd, 3rd, 4th, etc.
# Now with MALWARE!

import os

class str(str):
    """ str is an evil replacement for the built-in str class """
    def __str__(self):
        print('**** I AM IN YOR FILESYSTEM STEALING YOR SECRETS ****')
        os.system('ls -l') # Note: anywhere you see "ls -l" instead think: download virus and install it
        return super().__str__()

def ordinal(n: int) -> str:
    """Return an integer as its ordinal string (1st, 2nd, 3rd, etc.)."""
    if not isinstance(n, int):
        raise TypeError("n must be an integer")

    abs_n = abs(n)
    last_two = abs_n % 100

    if 11 <= last_two <= 13:
        suffix = "th"
    else:
        suffix = {1: "st", 2: "nd", 3: "rd"}.get(abs_n % 10, "th")

    if __name__ != '__main__':
        return str(f"{n}{suffix}")
    return f"{n}{suffix}"

if __name__ == '__main__':
    print(1,ordinal(1))    # "1st"
    print(2,ordinal(2))    # "2nd"
    print(3,ordinal(3))    # "3rd"
    print(4,ordinal(4))    # "4th"
    print(11,ordinal(11))   # "11th"
    print(22,ordinal(22))   # "22nd"
    print(113,ordinal(113))  # "113th"

In [4]:
!python3 ordinal_v2.py

1 1st
2 2nd
3 3rd
4 4th
11 11th
22 22nd
113 113th


In [5]:
from ordinal_v2 import ordinal

my_position=1
your_position=2
print( f"In this race, I am the {ordinal(my_position)} and you are the {ordinal(your_position)} at this time")

**** I AM IN YOR FILESYSTEM STEALING YOR SECRETS ****
total 32
-rw-r--r-- 1 peter peter 4641 Mar  3 17:14 Untitled.ipynb
drwxr-xr-x 2 peter peter 4096 Mar  3 17:12 __pycache__
-rw-r--r-- 1 peter peter  198 Mar  2 23:21 import_victim.py
-rw-r--r-- 1 peter peter  304 Mar  2 23:21 importbomb.py
-rw-r--r-- 1 peter peter  665 Mar  3 16:50 ordinal_v1.py
-rw-r--r-- 1 peter peter  912 Mar  3 16:51 ordinal_v2.py
-rw-r--r-- 1 peter peter  173 Mar  3 16:53 ordinal_victim.py
**** I AM IN YOR FILESYSTEM STEALING YOR SECRETS ****
total 32
-rw-r--r-- 1 peter peter 4641 Mar  3 17:14 Untitled.ipynb
drwxr-xr-x 2 peter peter 4096 Mar  3 17:12 __pycache__
-rw-r--r-- 1 peter peter  198 Mar  2 23:21 import_victim.py
-rw-r--r-- 1 peter peter  304 Mar  2 23:21 importbomb.py
-rw-r--r-- 1 peter peter  665 Mar  3 16:50 ordinal_v1.py
-rw-r--r-- 1 peter peter  912 Mar  3 16:51 ordinal_v2.py
-rw-r--r-- 1 peter peter  173 Mar  3 16:53 ordinal_victim.py
In this race, I am the 1st and you are the 2nd at this time


Aghhh! When I printed the "str" I got back from ordinal_v2 it ran shell commands on my system!
What happened to `str`, have a ruined the system `str` class?

In [8]:
s = ordinal(1)
print(s)
print( type(s))

**** I AM IN YOR FILESYSTEM STEALING YOR SECRETS ****
total 32
-rw-r--r-- 1 peter peter 7286 Mar  3 17:18 Untitled.ipynb
drwxr-xr-x 2 peter peter 4096 Mar  3 17:12 __pycache__
-rw-r--r-- 1 peter peter  198 Mar  2 23:21 import_victim.py
-rw-r--r-- 1 peter peter  304 Mar  2 23:21 importbomb.py
-rw-r--r-- 1 peter peter  665 Mar  3 16:50 ordinal_v1.py
-rw-r--r-- 1 peter peter  912 Mar  3 16:51 ordinal_v2.py
-rw-r--r-- 1 peter peter  173 Mar  3 16:53 ordinal_victim.py
1st
<class 'ordinal_v2.str'>


In [9]:
s = str("hello")
print(s)
print( type(s))

hello
<class 'str'>


No, Python knows the difference between the system `str` and the "bad" `ordinal_v2.str` classes.

# Why you never `from module import *`

In [10]:
from ordinal_v2 import *
s = str("hello")

print(s)
print(type(s))

**** I AM IN YOR FILESYSTEM STEALING YOR SECRETS ****
total 36
-rw-r--r-- 1 peter peter 8910 Mar  3 17:22 Untitled.ipynb
drwxr-xr-x 2 peter peter 4096 Mar  3 17:12 __pycache__
-rw-r--r-- 1 peter peter  198 Mar  2 23:21 import_victim.py
-rw-r--r-- 1 peter peter  304 Mar  2 23:21 importbomb.py
-rw-r--r-- 1 peter peter  665 Mar  3 16:50 ordinal_v1.py
-rw-r--r-- 1 peter peter  912 Mar  3 16:51 ordinal_v2.py
-rw-r--r-- 1 peter peter  173 Mar  3 16:53 ordinal_victim.py
hello
<class 'ordinal_v2.str'>


Oh no! `from ordinal_v2 import *` did replace the system `str` with the "bad" `str` because it put it in our local namespace, so it finds `ordinal_v2.str` before it finds the system `str`. So **DON'T DO THAT**

So am I safe if I'm careful with what functions and objects I use when I import from an untrusted module? Unfortunately no...

In [12]:
!cat importbomb.py

# This module does things as soon as it is imported, you don't have to call any functions

import os

print("**** I AM RUNNING IN YOUR SYSTEM BEFORE YOU EVEN CALL IMPORTED FUNCTIONS ****")
os.system("ls -l")

def first(l):
    return l[0]

def second(l):
    return l[1]

def rest(l):
    return l[1:]




In [14]:
import importbomb

**** I AM RUNNING IN YOUR SYSTEM BEFORE YOU EVEN CALL IMPORTED FUNCTIONS ****
total 36
-rw-r--r-- 1 peter peter 10187 Mar  3 17:24 Untitled.ipynb
drwxr-xr-x 2 peter peter  4096 Mar  3 17:26 __pycache__
-rw-r--r-- 1 peter peter   198 Mar  2 23:21 import_victim.py
-rw-r--r-- 1 peter peter   304 Mar  2 23:21 importbomb.py
-rw-r--r-- 1 peter peter   665 Mar  3 16:50 ordinal_v1.py
-rw-r--r-- 1 peter peter   912 Mar  3 16:51 ordinal_v2.py
-rw-r--r-- 1 peter peter   173 Mar  3 16:53 ordinal_victim.py


# eval() and exec() are dangerous!

Both `eval()` and `exec()` are able to run arbitrary code. If the parameters passed to `eval()` or `exec()` came from an untrusted source they can be abused.

In [2]:
print("Welcome to simple-calc, a simple Python calculator.")
# user_input = input("Enter an expression to evaluate:")
user_input = "1+2"
print(eval(user_input))

Welcome to simple-calc, a simple Python calculator.
3


In [6]:
import os
print("Welcome to simple-calc, a simple Python calculator.")
# user_input = input("Enter an expression to evaluate:")
user_input = "1+2,os.system('ls -l')"
print(eval(user_input))

Welcome to simple-calc, a simple Python calculator.
total 32
drwxr-xr-x 2 peter peter  4096 Mar  3 17:26 __pycache__
-rw-r--r-- 1 peter peter   304 Mar  2 23:21 importbomb.py
-rw-r--r-- 1 peter peter   665 Mar  3 16:50 ordinal_v1.py
-rw-r--r-- 1 peter peter   912 Mar  3 16:51 ordinal_v2.py
-rw-r--r-- 1 peter peter 14283 Mar  3 17:43 python-insecurity.ipynb
(3, 0)


## SQL Injection

In [6]:
username = input("Enter your username: ")

# Sql Alchemy using unsafe bare SQL
query = f"SELECT * FROM users WHERE username = '{username}'"
print( f'Generated SQL = {query}')
cursor.execute(query)

# Sql Alchemy using safe placeholders
query = "SELECT * FROM users WHERE username = ?"
cursor.execute(query, (username,))

# Hint, try: Robert'); DROP TABLE Students; --- comment

Enter your username:  Robert'); DROP TABLE Students; --- comment


Generated SQL = SELECT * FROM users WHERE username = 'Robert'); DROP TABLE Students; --- comment'


NameError: name 'cursor' is not defined

![xkcd-327](xkcd-327.png)

In [9]:
username = input("Enter your username to delete your account: ")

# Delete account using unsafe bare SQL
query = f"DELETE FROM users WHERE username = '{username}'"
print( f'Generated SQL = {query}')
cursor.execute(query)

# Delete account user safe placeholders
query = f"DELETE FROM users WHERE username = ?"
cursor.execute(query, (username,))

# Hint, try: peter' OR '1'='1' ---

Enter your username to delete your account:  peter' OR '1'='1' ---


Generated SQL = DELETE FROM users WHERE username = 'peter' OR '1'='1' ---'


NameError: name 'cursor' is not defined

OR '1'='1' ALL USERS DELETED!

## More...

- subprocess with shell=True
- unsafe temp file handling
- secrets in source code / .env leakage
- yaml.load with unsafe loaders
- trusting PATH / current directory imports
- accidental shadowing (random.py, os.py) causing import confusion

## Defensive checklist

- Treat imports as executable code.
- Never deserialize untrusted data with unsafe mechanisms.
- Do not execute strings as code.
- Minimize and audit dependencies.
- Validate types, inputs, and side effects explicitly.

## JH's contributions:
[on Discord](https://discord.com/channels/698267668918173827/1111141001818537985)
